# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

In [2]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [4]:
# Paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 3): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")

# Dates and locations
start_date = "11-01-2021"
end_date = "10-03-2025"
date_range = start_date + "--" + end_date
locations = "Antarctica,North America,South America"

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksi/Downloads/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" # "_Antarctica_North_America_South_America/"
andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])

serotype = "H5N1"
# genotypes = ["B3.13", "D1.1", "D1.3"]
genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]


## Downloading Data

In [5]:


# browser = "Chrome"
# sleep_time = "3"
# locations = "South America"
# start_date = "07-01-2025"
# end_date = "07-25-2025"



In [11]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Get files
            open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 

## De-Duplication

In [13]:
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Isolate, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Isolate")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments.head()

113317


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PX440412.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Galliformes,NaN,"Hicks,J.A., Stuber,T., Killian,M.L., Franzen,K...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8
1,PX440413.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Galliformes,NaN,"Hicks,J.A., Stuber,T., Killian,M.L., Franzen,K...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8
2,PX440414.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Galliformes,NaN,"Hicks,J.A., Stuber,T., Killian,M.L., Franzen,K...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8
3,PX440415.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Galliformes,NaN,"Hicks,J.A., Stuber,T., Killian,M.L., Franzen,K...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8
4,PX440416.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Galliformes,NaN,"Hicks,J.A., Stuber,T., Killian,M.L., Franzen,K...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8


In [14]:
# # De-duplicate from Andersen using SRA Accession

# # If even one SRA Accession in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
# andersen_sras = []
# # Grab files
# for dirpath, dirs, files in os.walk(andersen):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
#             sra_accessions = fasta_file["Identifier"]
#             for value in sra_accessions.values:
#                 if "SRR" in value:
#                     andersen_sras.append(value)
#             isolates = fasta_file["Isolate_Id"]
#             for value in isolates.values:
#                 andersen_sras.append(value)
#             partials = fasta_file["Partials"]
#             for value in partials.values:
#                 andersen_sras.append(value)
#     break 

# # Grab more files
# # for dirpath, dirs, files in os.walk(andersen_dedup):
# #     for file in files:
# #         file_name = os.path.join(dirpath, file)
# #         if ".fasta" in file_name:
# #             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
# #             sra_accessions = fasta_file["Identifier"]
# #             for value in sra_accessions.values:
# #                 if "SRR" in value:
# #                     andersen_sras.append(value)
            
# metadata_segments["Partials"] = metadata_segments["Isolate"].apply(lambda x: partial_isolate(x) if x == x else x)
# # Remove duplicates from Andersen
# for value in andersen_sras: # to remove
#     if "SRR" in value:
#         metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    
#     metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]
#     metadata_segments = metadata_segments[metadata_segments["Partials"] != value]
#     # print(value)
    
# print(len(metadata_segments))
# # metadata_segments
# # print(count)

## Add sequences to dataframe

In [15]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title | Accession

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
# sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
# sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-1][:-1])

# Extract segment number so that we can add the correct sequences to the correct sample
sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["Accession", "Segment"])

113317
                                         full_header  \
0  >Influenza A virus |USA: CA|25-022824-001-orig...   
1  >Influenza A virus |USA: CA|25-022824-001-orig...   
2  >Influenza A virus |USA: CA|25-022824-001-orig...   
3  >Influenza A virus |USA: CA|25-022824-001-orig...   
4  >Influenza A virus |USA: CA|25-022824-001-orig...   

                                            sequence   Accession  Segment  
0  ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...  PX440412.1        1  
1  ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...  PX440413.1        2  
2  ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...  PX440414.1        3  
3  ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...  PX440415.1        4  
4  ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...  PX440416.1        5  
99168


In [16]:
# Make sure H5N1 is the only serotype we have

metadata_segments["serotype"] = metadata_segments["full_header"].apply(lambda x: re.search(r'H.N.', x).group(0))

metadata_segments = metadata_segments[metadata_segments["serotype"] == serotype]

## Find genotype using old genoflu results or Andersen Lab genoflu output

In [17]:
''' Add old genoflu results manually from previous folder to this one (downloads_saved), named "output_old.tsv" '''

' Add old genoflu results manually from previous folder to this one (downloads_saved), named "output_old.tsv" '

In [18]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence,serotype
0,PX440412.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8,>Influenza A virus |USA: CA|25-022824-001-orig...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,H5N1
1,PX440413.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8,>Influenza A virus |USA: CA|25-022824-001-orig...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...,H5N1
2,PX440414.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8,>Influenza A virus |USA: CA|25-022824-001-orig...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,H5N1
3,PX440415.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8,>Influenza A virus |USA: CA|25-022824-001-orig...,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1
4,PX440416.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-08-12,2025-09-30,ssRNA(-),8,>Influenza A virus |USA: CA|25-022824-001-orig...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97715,OP691324.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Servicio Nacional de Sanidad, Inocuidad, y Cal...",Mexico,NaN,2022-10,2022-10-25,ssRNA(-),8,>Influenza A virus |Mexico: EdoMex|CPA-19638-2...,GGTTCACTCTGTCAAAATGGAGAACATAGTACTACTTCTTGCAATA...,H5N1
97716,OP691325.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Servicio Nacional de Sanidad, Inocuidad, y Cal...",Mexico,NaN,2022-10,2022-10-25,ssRNA(-),8,>Influenza A virus |Mexico: EdoMex|CPA-19638-2...,AAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCATGGCG...,H5N1
97717,OP691326.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Servicio Nacional de Sanidad, Inocuidad, y Cal...",Mexico,NaN,2022-10,2022-10-25,ssRNA(-),8,>Influenza A virus |Mexico: EdoMex|CPA-19638-2...,AAAGCAGGAGTTCAAAATGAATCCAAATCAAAAGATAACAACCATT...,H5N1
97718,OP691327.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Servicio Nacional de Sanidad, Inocuidad, y Cal...",Mexico,NaN,2022-10,2022-10-25,ssRNA(-),8,>Influenza A virus |Mexico: EdoMex|CPA-19638-2...,AAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAA...,H5N1


In [19]:
genoflu_old = pd.read_csv("output_old.tsv", delimiter="\t")
genoflu_old = genoflu_old.rename(columns={"sample":"Accession_Root"})
metadata_segments["Accession_Root"] = metadata_segments["Accession"].values[0].split(".")[0]

metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Accession_Root")
metadata_segments_new = metadata_segments.merge(genoflu_old,indicator = True, how='left').loc[lambda x : x['_merge']!='both']

print(len(metadata_segments_old))
print(len(metadata_segments_new))

0
97352


In [20]:
os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"})

metadata_segments_known = metadata_segments.merge(genoflu_andersen, how="inner", on="SRA_Accession")

metadata_segments_unknown = metadata_segments.merge(genoflu_andersen, indicator = True, how='left', on="SRA_Accession").loc[lambda x : x['_merge']!='both']

print(genoflu_andersen)

      SRA_Accession                 date       File Name  \
0       SRR30789580  2025-05-09_10-49-10  SRR30789580.fa   
1       SRR32973809  2025-05-09_10-52-30  SRR32973809.fa   
2       SRR32125669  2025-05-09_10-47-22  SRR32125669.fa   
3       SRR31597237  2025-05-09_10-47-24  SRR31597237.fa   
4       SRR31605135  2025-05-09_10-47-38  SRR31605135.fa   
...             ...                  ...             ...   
11163   SRR35319740  2025-09-12_06-16-28  SRR35319740.fa   
11164   SRR35319741  2025-09-12_06-16-27  SRR35319741.fa   
11165   SRR35319742  2025-09-12_06-16-22  SRR35319742.fa   
11166   SRR35319743  2025-09-12_06-16-26  SRR35319743.fa   
11167   SRR35319744  2025-09-12_06-16-22  SRR35319744.fa   

                                                Genotype  \
0      Not assigned: Only 0 segments >98.0% match fou...   
1                                                   D1.3   
2                                                  B3.13   
3                                      

In [21]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,serotype,Accession_Root,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,PX440412.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
1,PX440413.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
2,PX440414.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
3,PX440415.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
4,PX440416.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38179,PQ012131.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38180,PQ012132.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38181,PQ012133.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38182,PQ012134.1,GenBank,GCA_040780295.1,SRR29281472

In [22]:
metadata_segments_unknown

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Accession_Root,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,_merge
1624,PV780461.1,GenBank,GCA_052631165.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1625,PV780462.1,GenBank,GCA_052631165.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1626,PV780463.1,GenBank,GCA_052631165.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1627,PV780464.1,GenBank,GCA_052631165.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1628,PV780465.1,GenBank,GCA_052631165.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97347,OP691324.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
97348,OP691325.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
97349,OP691326.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
97350,OP691327.1,GenBank,GCA_039321425.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX440412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


## Create FASTA files of unknown genotypes using deduplicated sequences

In [23]:
# # Create 1 fasta file per header
# metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# # Create list of dataframes
# df_list = []
# for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
#     # Get smaller dataframe
#     df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
#     df = df.sort_values(by="Segment")
#     # Make sure there are 8 segments
#     if len(df) == 8:
#         # Forbidden characters in headers
#         for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
#             df["full_header"] = df["full_header"].apply(lambda x: x.replace(c, "_"))
#             df_list.append(df)

# # Make fasta files
# for df in df_list:
#     # Forbidden characters in file name 
#     title = df["Accession"].values[0].split(".")[0]
#     # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
#     #     title = title.replace(c, "_")
#     df_to_fasta(df, title + ".fasta", temp_files)

In [25]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # print(partial_header)
    # print(df["Segment"])
    # break 
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            # if c == "-":
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(df_list[0]["full_header"])

# Make fasta files
for df in df_list:
    # Forbidden characters in file name 
    # title = df["Accession"].values[0].split(".")[0]
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    # print(df)
    # print(segments)
    # break 
    # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    #     title = title.replace(c, "_")
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        # print(one_row["full_header"])
        # print(segment)
        # break 
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)
    # break 

61280    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61281    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61282    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61283    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61284    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61285    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61286    >Influenza_A_virus__USA__FL_22_024353_001_orig...
61287    >Influenza_A_virus__USA__FL_22_024353_001_orig...
Name: full_header, dtype: object


## Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "output.tsv" is in the downloads directory.



In [ ]:
'''
To run GenoFLU-multi, first change directories (and activate genoflu conda environment):

conda activate genoflu
cd GenoFLU-multi

And then call the python script:

python bin/genoflu-multi.py -f <FASTA_directory>
'''

'\n## In Ubuntu 22.04.3 LTS ##\n\nconda activate genoflu\n\n## Genoflu.py iteratively through all FASTAs in directory ##\n\nfor file in *.fasta; do genoflu.py -f "$file"; sleep 0.75; done\n\n## Concatenate output files ##\n\n awk \'(NR == 1) || (FNR > 1)\' *.tsv > output.tsv\n'

In [26]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,serotype,Accession_Root,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,PX440412.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
1,PX440413.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
2,PX440414.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
3,PX440415.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
4,PX440416.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38179,PQ012131.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38180,PQ012132.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38181,PQ012133.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38182,PQ012134.1,GenBank,GCA_040780295.1,SRR29281472

In [28]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("output.tsv", delimiter="\t")

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))
for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
    metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(c, "_"))

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(">", ""))

metadata_segments_unknown["Strain"] = metadata_segments_unknown["Partial_Header"]

metadata_genoflu = metadata_segments_unknown.merge(output_genoflu, how="left", on="Strain") 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill()

print(metadata_genoflu)

print(metadata_genoflu["Genotype"])


        Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0      PV780461.1        GenBank  GCA_052631165.1           NaN           NaN   
1      PV780462.1        GenBank  GCA_052631165.1           NaN           NaN   
2      PV780463.1        GenBank  GCA_052631165.1           NaN           NaN   
3      PV780464.1        GenBank  GCA_052631165.1           NaN           NaN   
4      PV780465.1        GenBank  GCA_052631165.1           NaN           NaN   
...           ...            ...              ...           ...           ...   
59163  OP691324.1        GenBank  GCA_039321425.1   SRR29149755  SAMN41462555   
59164  OP691325.1        GenBank  GCA_039321425.1   SRR29149755  SAMN41462555   
59165  OP691326.1        GenBank  GCA_039321425.1   SRR29149755  SAMN41462555   
59166  OP691327.1        GenBank  GCA_039321425.1   SRR29149755  SAMN41462555   
59167  OP691328.1        GenBank  GCA_039321425.1   SRR29149755  SAMN41462555   

         BioProject      Or

C:\Users\maksi\AppData\Local\Temp\ipykernel_25280\1771631458.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill()


# Concatenate with known genotypes

In [29]:
metadata_segments_known["Genotype"] = metadata_segments_known["Genotype_y"]

metadata_genoflu = pd.concat([metadata_genoflu, metadata_segments_known])

metadata_genoflu.columns

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'serotype', 'Accession_Root', 'date',
       'File Name', 'Genotype_y', 'Genotype List Used, >=98.0%_x',
       'Genotype Sample Title List_x', 'Genotype Percent Match List_x',
       'Genotype Mismatch List_x', 'Genotype Average Depth of Coverage List_x',
       '_merge', 'Partial_Header', 'Strain', 'Genotype',
       'Genotype List Used, >=98.0%_y', 'Genotype Sample Title List_y',
       'Genotype Percent Match List_y', 'Genotype Mismatch List_y',
       'Genotype Average Depth of Coverage List_y', 'Date run',

In [31]:
metadata_genoflu = metadata_genoflu[["Accession", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype", "Geo_Location", "full_header", "sequence", "serotype", "Segment", "File Name", "Partial_Header", "Strain"]]

metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] )
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1])

metadata_genoflu = metadata_genoflu.dropna(subset="SRA_Accession")
# metadata_genoflu = metadata_genoflu[~metadata_genoflu.duplicated(["Segment", "SRA_Accession"], keep=False).groupby(df["SRA_Accession"]).transform('sum').ge(8)]
# metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["SRA_Accession", "Segment"], keep="first", inplace=True)

metadata_genoflu[metadata_genoflu["Genotype"] == "B3.13"]

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,serotype,Segment,File Name,Partial_Header,Strain,genbank_name
26040,PQ790073.1,Influenza A virus (A/California/213/2024(H5N1)...,California,2024-12-06,SRR31864704,EPI_ISL_19628008,B3.13,USA: California,>Influenza A virus |USA: California|EPI_ISL_19...,GTAGATAATCACTCACTGAGTGACATCCACATCATGGCGTCTCAAG...,H5N1,5,NaN,Influenza_A_virus__USA__California_EPI_ISL_196...,Influenza_A_virus__USA__California_EPI_ISL_196...,A/California/213/2024
26041,PQ790074.1,Influenza A virus (A/California/213/2024(H5N1)...,California,2024-12-06,SRR31864704,EPI_ISL_19628008,B3.13,USA: California,>Influenza A virus |USA: California|EPI_ISL_19...,GTGACAAAAACATAATGGATTCCAACACTGTGTTAAGCTTTCAGGT...,H5N1,8,NaN,Influenza_A_virus__USA__California_EPI_ISL_196...,Influenza_A_virus__USA__California_EPI_ISL_196...,A/California/213/2024
26042,PQ790075.1,Influenza A virus (A/California/213/2024(H5N1)...,California,2024-12-06,SRR31864704,EPI_ISL_19628008,B3.13,USA: California,>Influenza A virus |USA: California|EPI_ISL_19...,TACTGATTCAAAATGGAAGACTTTGTGCGACAATGCTTCAATCCAA...,H5N1,3,NaN,Influenza_A_virus__USA__California_EPI_ISL_196...,Influenza_A_virus__USA__California_EPI_ISL_196...,A/California/213/2024
26043,PQ790076.1,Influenza A virus (A/California/213/2024(H5N1)...,California,2024-12-06,SRR31864704,EPI_ISL_19628008,B3.13,USA: California,>Influenza A virus |USA: California|EPI_ISL_19...,AGTTCAAAATGAATCCAAATCAAAAGATAACAACCATTGGATCAAT...,H5N1,6,NaN,Influenza_A_virus__USA__California_EPI_ISL_196...,Influenza_A_virus__USA__California_EPI_ISL_196...,A/California/213/2024
26044,PQ790077.1,Influenza A virus (A/California/213/2024(H5N1)...,California,2024-12-06,SRR31864704,EPI_ISL_19628008,B3.13,USA: California,>Influenza A virus |USA: California|EPI_ISL_19...,GGTTCACTCTGTCAAAATGGAGAACATAGTGCTTCTTCTTGCAATA...,H5N1,4,NaN,Influenza_A_virus__USA__California_EPI_ISL_196...,Influenza_A_virus__USA__California_EPI_ISL_196...,A/California/213/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38179,PQ012131.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38180,PQ012132.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38181,PQ012133.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAATCCAAATCAAAAGATAACAACCATTGGATCAATCTGTATGG...,H5N1,6,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38182,PQ012134.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024


We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [33]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['cormorant', 'black-legged kittiwake', 'turkey vulture', 'gull', 'duck', 'american robin', 'snowy plover', 'louisiana', 'rough-legged hawk', 'black-crowned night heron', 'quail', 'american wood stork', 'horned grebe', 'american kestrel', 'snow goose', 'western gull', 'magpie', 'swainsons hawk', 'parrot', 'pigeon', 'ring-necked duck', 'common goldeneye', 'backyard bird', 'gannet', 'savannah cat', 'red-shouldered hawk', 'short-billed gull', 'alpaca', 'ring-billed gull', 'herring gull', 'swan', 'chicken', 'broad-winged hawk', 'green heron', 'parasitic jaeger', 'great egret', 'tiger', 'muscovy duck', 'eurasian collared dove', 'sandwich tern', 'cattle', 'western grebe', 'owl', 'lesser snow goose white-morph', 'mute swan', 'peregrine falcon', 'bufflehead', 'green-winged teal', 'redhead', 'pintail', 'dunlin', 'western sandpiper', 'gadwall', 'mallard duck', 'sanderling', 'white-winged scoter', 'teal', 'baikal teal', 'red-tailed hawk', 'american crow', 'black vulture', 'willet', 'waterfowl', '

In [34]:
metadata_genoflu

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,serotype,Segment,File Name,Partial_Header,Strain,genbank_name
26008,PQ824938.1,Influenza A virus (A/chicken/Washington/W24285...,chicken,2024-10-11,SRR31864705,W242850087-1,D1.1,USA: Washington,>Influenza A virus |USA: Washington|W242850087...,ATGGATTCCAACACTGTGTCAAGCTTTCAGGTAGACTGCTTTCTTT...,H5N1,8,NaN,Influenza_A_virus__USA__Washington_W242850087_...,Influenza_A_virus__USA__Washington_W242850087_...,A/chicken/Washington/W242850087-1/2024
26009,PQ824939.1,Influenza A virus (A/chicken/Washington/W24285...,chicken,2024-10-11,SRR31864705,W242850087-1,D1.1,USA: Washington,>Influenza A virus |USA: Washington|W242850087...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,NaN,Influenza_A_virus__USA__Washington_W242850087_...,Influenza_A_virus__USA__Washington_W242850087_...,A/chicken/Washington/W242850087-1/2024
26010,PQ824940.1,Influenza A virus (A/chicken/Washington/W24285...,chicken,2024-10-11,SRR31864705,W242850087-1,D1.1,USA: Washington,>Influenza A virus |USA: Washington|W242850087...,ATGAATCCAAATCAAAAGATAATAACTATCGGGTCAATCTGCATGG...,H5N1,6,NaN,Influenza_A_virus__USA__Washington_W242850087_...,Influenza_A_virus__USA__Washington_W242850087_...,A/chicken/Washington/W242850087-1/2024
26011,PQ824941.1,Influenza A virus (A/chicken/Washington/W24285...,chicken,2024-10-11,SRR31864705,W242850087-1,D1.1,USA: Washington,>Influenza A virus |USA: Washington|W242850087...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,NaN,Influenza_A_virus__USA__Washington_W242850087_...,Influenza_A_virus__USA__Washington_W242850087_...,A/chicken/Washington/W242850087-1/2024
26012,PQ824942.1,Influenza A virus (A/chicken/Washington/W24285...,chicken,2024-10-11,SRR31864705,W242850087-1,D1.1,USA: Washington,>Influenza A virus |USA: Washington|W242850087...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,H5N1,4,NaN,Influenza_A_virus__USA__Washington_W242850087_...,Influenza_A_virus__USA__Washington_W242850087_...,A/chicken/Washington/W242850087-1/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38179,PQ012131.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38180,PQ012132.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38181,PQ012133.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAATCCAAATCAAAAGATAACAACCATTGGATCAATCTGTATGG...,H5N1,6,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38182,PQ012134.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024


In [35]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        "USA"
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


# Make new labels
names = ">" + metadata_genoflu["SRA_Accession"].apply(lambda x: x.split(",")[0]) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Geo_Location_Abrv"])


26008    USA-WA
26009    USA-WA
26010    USA-WA
26011    USA-WA
26012    USA-WA
          ...  
38179       USA
38180       USA
38181       USA
38182       USA
38183       USA
Name: Geo_Location_Abrv, Length: 71344, dtype: object


## Rename segments and make complete FASTA files

In [36]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

B3.7_PB2
A3_PB2
B3.5_PB2
B3.6_PB2
D1.3_PB2
B3.13_PB2
B3.2_PB2
D1.1_PB2
B3.7_PB1
A3_PB1
B3.5_PB1
B3.6_PB1
D1.3_PB1
B3.13_PB1
B3.2_PB1
D1.1_PB1
B3.7_PA
A3_PA
B3.5_PA
B3.6_PA
D1.3_PA
B3.13_PA
B3.2_PA
D1.1_PA
B3.7_HA
A3_HA
B3.5_HA
B3.6_HA
D1.3_HA
B3.13_HA
B3.2_HA
D1.1_HA
B3.7_NP
A3_NP
B3.5_NP
B3.6_NP
D1.3_NP
B3.13_NP
B3.2_NP
D1.1_NP
B3.7_NA
A3_NA
B3.5_NA
B3.6_NA
D1.3_NA
B3.13_NA
B3.2_NA
D1.1_NA
B3.7_MP
A3_MP
B3.5_MP
B3.6_MP
D1.3_MP
B3.13_MP
B3.2_MP
D1.1_MP
B3.7_NS
A3_NS
B3.5_NS
B3.6_NS
D1.3_NS
B3.13_NS
B3.2_NS
D1.1_NS


C:\Users\maksi\AppData\Local\Temp\ipykernel_25280\480553321.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)


In [38]:
print(metadata_genoflu[metadata_genoflu["Genotype"] == "B3.2"])

        Accession                                      GenBank_Title  \
26944  PQ701222.1  Influenza A virus (A/American Blue-winged teal...   
26945  PQ701223.1  Influenza A virus (A/American Blue-winged teal...   
26946  PQ701224.1  Influenza A virus (A/American Blue-winged teal...   
26947  PQ701225.1  Influenza A virus (A/American Blue-winged teal...   
26948  PQ701226.1  Influenza A virus (A/American Blue-winged teal...   
...           ...                                                ...   
31251  PQ687455.1  Influenza A virus (A/chicken/PA/24-031844-002-...   
31252  PQ687456.1  Influenza A virus (A/chicken/PA/24-031844-002-...   
31253  PQ687457.1  Influenza A virus (A/chicken/PA/24-031844-002-...   
31254  PQ687458.1  Influenza A virus (A/chicken/PA/24-031844-002-...   
31255  PQ687459.1  Influenza A virus (A/chicken/PA/24-031844-002-...   

                            Host Collection_Date SRA_Accession  \
26944  american blue-winged teal      2022-09-11   SRR31864704   
269

In [39]:
# Create FASTA files

os.chdir(complete_files)

# names = []

for df in segment_genotype_dfs:
    print(df)
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0:
        # print(df)
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
    output_file.close()

        Accession                                      GenBank_Title  \
34432  PQ708710.1  Influenza A virus (A/Canada goose/OR/23-031491...   
18456  PV338481.1  Influenza A virus (A/cat/MT/24-006142-001-orig...   
25096  PQ827602.1  Influenza A virus (A/Canada Goose/IN/24-005146...   
25112  PQ827618.1  Influenza A virus (A/Canada Goose/MO/24-006385...   
25136  PQ827650.1  Influenza A virus (A/Crow/IL/24-004479-001-ori...   
25144  PQ827658.1  Influenza A virus (A/Crow/IL/24-004479-002-ori...   
25464  PQ828066.1  Influenza A virus (A/Western Gull/CA/24-004708...   

               Host Collection_Date SRA_Accession  \
34432  canada goose      2023-10-11   SRR31864704   
18456           cat      2023-12-05   SRR32654216   
25096  canada goose      2024-02-14   SRR28834886   
25112  canada goose      2024-02-22   SRR28834902   
25136          crow      2024-02-08   SRR28834920   
25144          crow      2024-02-08   SRR28834909   
25464  western gull      2024-01-30   SRR28834951   

In [43]:
# # Concatenate with new Andersen sequences

# os.chdir(combined_files)

# # andersen = home + "Andersen/complete/" + date_range + "/"

# # Andersen files
# filenames_andersen = []
# for genotype in genotypes:
#     # print(gisaid_andersen + genotype.replace(".", "_") + "/")
#     # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
#     for dirpath, dirs, files in os.walk(andersen): # Find the fasta file
#         for file in files:
#             file_name = os.path.join(dirpath, file) # Get file name
#             filenames_andersen.append(file_name)
#         break 

# # NCBI Virus files
# filenames_ncbi = []
# for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file) # Get file name
#         filenames_ncbi.append(file_name)
#     break 

# print(filenames_ncbi)

# common_genotypes = set()
# # Concatenate the two -- should not have any overlap due to dates and deduplication 
# for a_file in filenames_andersen:
#     partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
#     for nv_file in filenames_ncbi:
#         partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
#         if partial_filename_a == partial_filename_nv:
#             common_genotypes.add(partial_filename_a)
#             filenames = [a_file, nv_file]
#             with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile:
#                         for line in infile:
#                             outfile.write(line)
#                         infile.close()
#                 outfile.close()

# print(common_genotypes)

# for a_file in filenames_andersen:
#     partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
#     # If genotype not found in NCBI Virus, include it as well
#     if partial_filename_a not in common_genotypes:
#         with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile2:
#             with open(a_file) as infile2:
#                 for line in infile2:
#                     outfile2.write(line)
#                 infile2.close()
#             outfile2.close()